In [37]:
import kagglehub
import shutil, pathlib

CSV_PATH = "data/sudoku.csv"

if pathlib.Path("data").exists() and pathlib.Path(CSV_PATH).exists():
    print("Data already exists")
else:
    print("Downloading data")
    path = kagglehub.dataset_download("bryanpark/sudoku")
    dest = pathlib.Path("data")
    dest.mkdir(exist_ok=True)
    shutil.copy(pathlib.Path(path) / "sudoku.csv", dest / "sudoku.csv")
    print(f"Copied to {dest / 'sudoku.csv'}")



Data already exists


In [ ]:
# Masking Policies
import torch

def train_mask_random_k(idx: int, max_k: int = 20) -> torch.Tensor:
    k = torch.randint(1, max_k + 1, (1,)).item()
    return torch.randperm(81)[:k]                       # fresh randomness every access

def val_mask_deterministic_k(idx: int, max_k: int = 20) -> torch.Tensor:
    g = torch.Generator().manual_seed(idx)              # same cells for same idx, forever
    k = 1 + idx % max_k
    return torch.randperm(81, generator=g)[:k]

def single_blank_mask(idx: int) -> torch.Tensor:        # tonight's original val policy
    return torch.tensor([idx % 81])

def two_blank_mask(idx: int) -> torch.Tensor:           # the distractor experiment
    target = idx % 81
    distractor = (target + 1 + (idx * 37 + 11) % 80) % 81
    return torch.tensor([target, distractor])

In [ ]:

from typing import Callable
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset
from enum import Enum

class DatasetSplit(Enum):
    ALL = 0
    TRAINING = 1
    VALIDATION = 2


class SudokuDataset(Dataset):
    def __init__(
        self,
        csv_file_path: str,
        split: DatasetSplit,
        mask_fn: Callable[[int], torch.Tensor],
        seed: int = 0,
        val_size: int = 10_000,
    ):
        with open(csv_file_path, 'rb') as f:
            f.readline()
            buf = np.frombuffer(f.read(), dtype=np.uint8).reshape(-1, 164)
        all_solutions = (buf[:, 82:163] - ord('0')).astype(np.int8)

        g = torch.Generator().manual_seed(seed)
        perm = torch.randperm(len(all_solutions), generator=g)
        match split:
            case DatasetSplit.TRAINING:
                idxs = perm[val_size:]
            case DatasetSplit.VALIDATION:
                idxs = perm[:val_size]
            case DatasetSplit.ALL:
                idxs = perm
        self.solutions = all_solutions[idxs.numpy()]
        self.mask_fn = mask_fn

    def __len__(self):
        return len(self.solutions)

    def __getitem__(self, idx: int):
        puzzle = self.solutions[idx].copy()
        cells = self.mask_fn(idx)

        target_cell = cells[0].item()
        original_answer = torch.tensor(puzzle[target_cell] - 1, dtype=torch.long)

        puzzle[cells.numpy()] = 0
        return torch.from_numpy(puzzle), torch.tensor(target_cell), original_answer

def encode_batch(digits: torch.Tensor, query_cells: torch.Tensor) -> torch.Tensor:
    board = F.one_hot(digits.long(), num_classes=10).float().flatten(1)   # (B, 810)
    query = F.one_hot(query_cells.long(), num_classes=81).float()         # (B, 81)
    return torch.cat([board, query], dim=1)                                # (B, 891)

In [39]:
import torch
from torch import nn, Tensor

class SudokuMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_layer = nn.Linear(891,256)
        self.activation = nn.ReLU()
        self.output_layer = nn.Linear(256,9)
    
    def forward(self, x: Tensor):
        x = self.hidden_layer(x)
        x = self.activation(x)
        return self.output_layer(x)

test_model = SudokuMLP()
test_input = torch.randn(4, 891)
assert test_model(test_input).shape == torch.Size([4,9])


In [40]:
from torch.utils.data import DataLoader

train_dataset     = SudokuDataset(CSV_PATH, DatasetSplit.TRAINING,   mask_fn=train_mask_random_k)
val_dataset       = SudokuDataset(CSV_PATH, DatasetSplit.VALIDATION, mask_fn=val_mask_deterministic_k)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1024, shuffle=False)

In [41]:
import torch.nn as nn

def evaluate(model: SudokuMLP, val_loader: DataLoader, loss_fn: nn.CrossEntropyLoss):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for digits, query, y in val_loader:
            x = encode_batch(digits, query)
            logits = model(x)
            loss = loss_fn(logits, y)

            batch_size = x.shape[0]

            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_examples += batch_size

    model.train()
    return total_loss / total_examples, total_correct / total_examples

In [43]:
import mlflow
import mlflow.data
import torch.nn as nn
import torch

mlflow.set_tracking_uri("http://gaming-pc:5000/")
mlflow.set_experiment("sudoku-solver")

NUM_EPOCHS = 10

model = SudokuMLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_ds_for_mlflow = mlflow.data.from_numpy(
    features=train_dataset.solutions,
    source="data/sudoku.csv",
    name="sudoku-1m-train-split"
)

with mlflow.start_run():
    mlflow.log_params({
        "hidden_size": 256,
        "num_hidden_layers": 1,
        "lr": 1e-3,
        "optimizer": "adam",
        "batch_size": 256,
        "num_epochs": NUM_EPOCHS,
        "val_size": 10_000,
        "split_seed": 0,
        "mask_strategy_train": "uniform_random_k1-20",
        "mask_strategy_val": "seeded_randperm_k1-20",
        "encoding": "onehot_810_plus_query81",
    })
    mlflow.log_input(train_ds_for_mlflow, context="training")

    # verify initial perf
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    mlflow.log_metric("val_loss", val_loss, step=0)
    mlflow.log_metric("val_acc", val_acc, step=0)

    step = 0
    for epoch in range(NUM_EPOCHS):
        model.train()
        for digits, query, y in train_loader:
            x = encode_batch(digits, query)
            # forward
            logits = model(x)
            # loss
            loss = loss_fn(logits, y)
            # zero_grad
            optimizer.zero_grad()
            # backward
            loss.backward()
            # step
            optimizer.step()

            step += 1
            if step % 100 == 0:
                mlflow.log_metric("train_loss", loss.item(), step=step)
            if step % 1000 == 0:
                val_loss, val_acc = evaluate(model, val_loader, loss_fn)
                mlflow.log_metric("val_loss", val_loss, step=step)
                mlflow.log_metric("val_acc", val_acc, step=step)

    # final eval
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    mlflow.log_metric("val_loss", val_loss, step=step)
    mlflow.log_metric("val_acc", val_acc, step=step)

    mlflow.pytorch.log_model(
        model, 
        name="model",
        input_example=torch.randn(1, 891).numpy(),
    )



c:\Users\brand\Projects\sudokuSolver\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/08/16 10:06:44 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\brand\Projects\sudokuSolver
W0816 10:06:44.294000 7636 Lib\site-packages\torch\_export\non_strict_utils.py:656] dimension inputs['x'].shape[0] 0/1 specialized; Dim.AUTO was specified along with a sample input with hint = 1.
2026/08/16 10:06:44 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\brand\Projects\sudokuSolver
2026/08/16 10:06:44 INFO mlflow.utils.environment: Detected uv project at c:\Users\brand\Projects\sudokuSolver. Attempting to export requirements via 'uv export'.
2026/

🏃 View run redolent-vole-992 at: http://gaming-pc:5000/#/experiments/6/runs/5a316ce463a94a07adb51f4b6285d99c
🧪 View experiment at: http://gaming-pc:5000/#/experiments/6


In [44]:
import mlflow

two_blank_val     = SudokuDataset(CSV_PATH, DatasetSplit.VALIDATION, mask_fn=two_blank_mask)
two_blank_val_loader   = DataLoader(two_blank_val,   batch_size=1024, shuffle=False)

ds_for_mlflow = mlflow.data.from_numpy(
    features=two_blank_val.solutions,
    source="data/sudoku.csv",
    name="sudoku-1m-val-split"
)

with mlflow.start_run():
    mlflow.log_input(ds_for_mlflow, context="val")

    val_loss, val_acc = evaluate(model, two_blank_val_loader, loss_fn)
    mlflow.log_metric("val_loss", val_loss, step=step)
    mlflow.log_metric("val_acc", val_acc, step=step)

c:\Users\brand\Projects\sudokuSolver\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


🏃 View run nebulous-loon-972 at: http://gaming-pc:5000/#/experiments/6/runs/be8541e564c749c182cff9af45d72d7a
🧪 View experiment at: http://gaming-pc:5000/#/experiments/6
